# Снижение активности по картам: кто, насколько и за счёт чего

**Автор:** Таболин Владислав Андреевич

**Период данных:** 2024-01-01 — 2025-06-30

**Подготовлено для:** департамент розничных продуктов, банк «Меридиан»

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Markdown, display

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "font.size": 9})
GROUP_NAMES = {"G1_развлечения": "G1. Ушли из развлечений",
               "G2_массовый_без_приложения": "G2. Массовый без приложения",
               "G3_массовый_с_приложением": "G3. Массовый с приложением"}

clean = pd.read_parquet("clean_data.parquet")
cm = pd.read_parquet("client_metrics.parquet")
log = pd.read_csv("cleaning_log.csv")
hyp = pd.read_csv("hypotheses.csv")
excl = pd.read_csv("exclusions.csv")
camp = pd.read_csv("campaign_list.csv")

MONTHS = [str(p) for p in pd.period_range("2024-01", "2025-06", freq="M")]
ok = cm[cm["measurable"]]
mass = ok["segment"] == "Массовый"
grp = camp["group"].value_counts()
gm = ok.set_index("client_id").loc[camp["client_id"]].assign(group=camp["group"].values)
med = gm.groupby("group")[["rate_base", "rate_eval", "chg", "ent_chg", "other_chg"]].median()

def pct(x): return f"{x * 100:+.0f}%"
monthly = clean.groupby("month").size().reindex(MONTHS)
h1_24, h1_25 = monthly[:6].mean(), monthly[12:].mean()
seg_app = ok.groupby(["segment", "has_mobile_app"])["chg"].median()

## 1. Главное

In [ ]:
display(Markdown(f"""
1. **В кампанию идут {len(camp)} клиентов в трёх группах** ({', '.join(f'{GROUP_NAMES[g][:2]} — {n}' for g, n in grp.sort_index().items())}),
   все с согласием на коммуникацию. Это укладывается в бюджет 2000 и в минимум 150 на группу.
2. **Падение сосредоточено в массовом сегменте.** У массовых клиентов без приложения медиана изменения
   числа операций в месяц {pct(seg_app[('Массовый', False)])}, с приложением {pct(seg_app[('Массовый', True)])}.
   У остальных сегментов от {pct(seg_app.drop('Массовый').min())} до {pct(seg_app.drop('Массовый').max())},
   из них около −7% сезонные (n = {len(ok)} клиентов с измеримой активностью, 2025 к 2024).
3. **Отдельная причина — развлечения.** С января 2025 года траты в категории «Развлечения и досуг» обвалились
   во всех сегментах и городах. У {grp['G1_развлечения']} клиентов группы G1 эта категория упала на {pct(med.loc['G1_развлечения', 'ent_chg'])}
   (медиана), остальные траты почти не изменились ({pct(med.loc['G1_развлечения', 'other_chg'])}).
4. **Пластик не «умер», его заменил токен.** Операции по карте в 2025 году — {clean[clean.window == 'оценка 2025'].channel.eq('Карта').mean():.0%}
   от всех против {clean[clean.window == 'база 2024'].channel.eq('Карта').mean():.0%} в 2024-м. Это смена способа оплаты, а не падение, и в кампанию такие клиенты не берутся.
5. **Казань — не причина.** Казань выглядит худшим городом только потому, что там 87% клиентов массового сегмента;
   внутри сегмента она не отличается от других городов.
"""))

## 2. Контекст

Клиенты банка за последние полгода реже пользуются картами. Департамент готовит кампанию по возврату активности
на 2000 клиентов. Коммуникация готовится под группу, поэтому нужно 3–5 групп не меньше 150 человек,
каждая описана правилом по признакам клиента и понятной причиной падения. Отчёт отвечает на вопрос,
к кому идти, с каким предложением и кого не трогать.

## 3. Данные и ограничения

In [ ]:
n_cl = len(cm)
display(Markdown(f"""
**Период:** база — 2024 год (12 мес.), оценка — январь–июнь 2025 года (6 мес.). **Данные:** {n_cl} действующих клиентов
на 30.06.2025 и {len(clean)} успешных операций после очистки. Исключены дубли, операции вне периода,
статусы «отклонено» и «возврат», а также операции {excl.loc[0, 'клиентов']} клиентов, которых нет в справочнике действующих
(они уже ушли из банка). Сводка исправлений — в приложении.

**Метрика активности.** Среднее **число успешных операций по карте в месяц**, по всем способам оплаты
(пластик, токен, СБП, онлайн-банк), считается с месяца после открытия текущей карты. Изменение — отношение
среднемесячного числа операций в январе–июне 2025 года к 2024 году минус 1. Падением считаем −30% и больше:
это втрое больше сезонного эффекта (первое полугодие ≈ −7% к среднему за год). Измерить удалось у {len(ok)}
клиентов: у {excl.loc[1, 'клиентов']} карта открыта после 01.12.2024, у {excl.loc[2, 'клиентов']} меньше одной операции в месяц.
**Метрика не увидит** снижения сумм при той же частоте, перераспределения трат между категориями
и падения только в последние недели окна (среднее за 6 месяцев его сглаживает).

**Чего нет в данных:** операций клиента в других банках, причин закрытия карт у ушедших, условий кэшбэка
и акций, истории изменения сегмента; доход и оценка сервиса известны не у всех (оценка — у 41% клиентов).
"""))

## 4. Находки

### 4.1. Общее число операций в первом полугодии 2025 года на 11% ниже, чем годом раньше; часть провала в январе — сезонная

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 2.6))
ax.plot(range(1, 13), monthly[:12].values / 1000, marker="o", label="2024")
ax.plot(range(1, 7), monthly[12:].values / 1000, marker="o", label="2025")
ax.set(xticks=range(1, 13), xlabel="месяц года", ylabel="тыс. операций в месяц",
       title=f"Успешные операции действующих клиентов по месяцам, n = {len(clean)} операций")
ax.legend(); plt.tight_layout(); plt.show()
display(Markdown(f"В среднем за месяц: январь–июнь 2024 года — {h1_24:,.0f}, январь–июнь 2025 года — {h1_25:,.0f} операций "
                 f"({pct(h1_25 / h1_24 - 1)}). Декабрьский пик и январский провал повторяются каждый год, поэтому сравниваем "
                 f"среднемесячные значения и ставим порог падения с запасом над сезонностью.".replace(",", " ")))

### 4.2. Падение сосредоточено в массовом сегменте, особенно у клиентов без мобильного приложения

In [ ]:
tab = ok.groupby(["segment", "has_mobile_app"])["chg"].agg(["median", "size"]).reset_index()
tab["приложение"] = tab["has_mobile_app"].map({True: "есть", False: "нет"})
order = ["Массовый", "Зарплатный", "Премиальный", "Private"]
fig, ax = plt.subplots(figsize=(7.5, 2.6))
sns.barplot(data=tab, x="segment", y=tab["median"] * 100, hue="приложение", order=order, ax=ax)
for c in ax.containers: ax.bar_label(c, fmt="%.0f%%", fontsize=8)
ax.axhline(-30, color="red", ls="--", lw=1)
ax.set(xlabel="сегмент", ylabel="медиана изменения, %",
       title=f"Изменение операций в месяц, 2025 к 2024, n = {len(ok)} клиентов")
plt.tight_layout(); plt.show()
n_tab = tab.set_index(["segment", "приложение"])["size"]
display(Markdown(f"Массовый сегмент: без приложения n = {n_tab[('Массовый', 'нет')]}, с приложением n = {n_tab[('Массовый', 'есть')]}. "
                 "У массовых клиентов падение идёт по всем категориям сразу. Приложение смягчает падение, "
                 "но не останавливает его. Красная линия — порог падения −30%."))

### 4.3. Траты на развлечения обвалились в январе 2025 года во всех сегментах

In [ ]:
idx = clean.assign(cat=np.where(clean["category"] == "Развлечения и досуг", "Развлечения и досуг", "остальные 8 категорий")) \
    .pivot_table(index="month", columns="cat", values="txn_id", aggfunc="count").reindex(MONTHS)
idx = idx / idx.iloc[:12].mean()
fig, ax = plt.subplots(figsize=(7.5, 2.6))
idx.plot(ax=ax, marker="o")
ax.axvline(11.5, color="grey", ls=":")
ax.set(xlabel="месяц", ylabel="к среднему за 2024 (1,0)", title="Индекс числа операций по категориям")
ax.set_xticks(range(len(MONTHS))); ax.set_xticklabels(MONTHS, rotation=60)
plt.tight_layout(); plt.show()
e25 = idx.loc[MONTHS[12:], "Развлечения и досуг"].mean(); o25 = idx.loc[MONTHS[12:], "остальные 8 категорий"].mean()
display(Markdown(f"В январе–июне 2025 года операций в развлечениях в среднем {e25:.2f} от уровня 2024 года, "
                 f"в остальных категориях {o25:.2f}. Обвал одновременный и общий, поэтому причина, скорее всего, "
                 "на стороне категории (условия кэшбэка или партнёрская сеть), а не у клиентов."))

### 4.4. Клиенты не перестали платить картой, а перешли на оплату смартфоном

In [ ]:
ch = clean.pivot_table(index="month", columns="channel", values="txn_id", aggfunc="count").reindex(MONTHS)
ch = ch / ch.iloc[:12].mean()
fig, ax = plt.subplots(figsize=(7.5, 2.6))
ch.plot(ax=ax, marker=".")
ax.axvline(11.5, color="grey", ls=":")
ax.set(xlabel="месяц", ylabel="к среднему за 2024 (1,0)", title="Индекс числа операций по способу оплаты")
ax.set_xticks(range(len(MONTHS))); ax.set_xticklabels(MONTHS, rotation=60)
plt.tight_layout(); plt.show()
display(Markdown(f"В 2025 году операций по пластиковой карте {ch.loc[MONTHS[12:], 'Карта'].mean():.2f} от уровня 2024 года, "
                 f"по токену {ch.loc[MONTHS[12:], 'Токен'].mean():.2f}. Поэтому метрика считает все способы оплаты. "
                 f"Провал «Онлайн-банка» в апреле–мае ({ch.loc['2025-04', 'Онлайн-банк']:.2f} и {ch.loc['2025-05', 'Онлайн-банк']:.2f}) "
                 f"восстановился в июне ({ch.loc['2025-06', 'Онлайн-банк']:.2f}). Это похоже на технический сбой."))

### 4.5. Казань хуже других городов только из-за состава клиентов

In [ ]:
city_all = ok.groupby("city")["chg"].median()
city_mass = ok[mass].groupby("city")["chg"].median()
share_mass = cm.groupby("city")["segment"].apply(lambda s: (s == "Массовый").mean())
d = pd.DataFrame({"все клиенты": city_all, "только массовый сегмент": city_mass}) * 100
fig, ax = plt.subplots(figsize=(7.5, 2.6))
d.sort_values("все клиенты").plot.bar(ax=ax)
ax.set(xlabel="", ylabel="медиана изменения, %", title=f"Изменение операций в месяц по городам, n = {len(ok)} клиентов")
ax.tick_params(axis="x", rotation=0, labelsize=8)
plt.tight_layout(); plt.show()
display(Markdown(f"По всем клиентам Казань {pct(city_all['Казань'])} при {pct(city_all.drop('Казань').median())} в среднем по другим городам. "
                 f"Внутри массового сегмента Казань {pct(city_mass['Казань'])}, другие города {pct(city_mass.drop('Казань').min())}…{pct(city_mass.drop('Казань').max())}. "
                 f"Доля массовых клиентов: Казань {share_mass['Казань']:.0%}, другие города {share_mass.drop('Казань').min():.0%}–{share_mass.drop('Казань').max():.0%}. "
                 "Отдельную группу по Казани не выделяем."))

## 5. Группы для кампании

In [ ]:
rules = {
    "G1_развлечения": ("клиенты, регулярно платившие за развлечения (≥0,5 операции/мес. в 2024)",
                       "развлечения −70% и больше, остальные траты не более −30%"),
    "G2_массовый_без_приложения": ("массовый сегмент, нет мобильного приложения", "падение −30% и больше по всем категориям"),
    "G3_массовый_с_приложением": ("массовый сегмент, приложение есть", "падение −30% и больше по всем категориям"),
}
t = pd.DataFrame([{"группа": GROUP_NAMES[g], "кто": rules[g][0], "что произошло": rules[g][1],
                   "клиентов": grp[g],
                   "операций/мес. 2024 → 2025 (медиана)": f"{med.loc[g, 'rate_base']:.1f} → {med.loc[g, 'rate_eval']:.1f}"}
                  for g in rules])
display(t.style.hide(axis="index"))

fig, ax = plt.subplots(figsize=(7.5, 2.6))
M = clean.groupby(["client_id", "month"]).size().unstack(fill_value=0).reindex(columns=MONTHS, fill_value=0)
for g in rules:
    ids = camp.loc[camp["group"] == g, "client_id"]
    traj = M.reindex(ids).fillna(0).sum() / ok.set_index("client_id").loc[ids, "rate_base"].sum()
    ax.plot(MONTHS, traj.values, marker=".", label=f"{GROUP_NAMES[g]} (n = {len(ids)})")
ax.axvline(11.5, color="grey", ls=":")
ax.set(xlabel="месяц", ylabel="к уровню 2024 (1,0)", title="Активность групп по месяцам")
ax.tick_params(axis="x", rotation=60); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
display(Markdown(f"""
Клиент, подходящий под несколько правил, попадает в первую подходящую группу в порядке G1 → G2 → G3:
сначала самая конкретная причина. Пересечений немного (G1 с G2 — 21 клиент, G1 с G3 — 37).

**G1. Ушли из развлечений ({grp['G1_развлечения']} клиентов).** Клиенты всех сегментов, активные в целом,
но с января 2025 года почти перестали платить картой в развлечениях: {pct(med.loc['G1_развлечения', 'ent_chg'])} в категории
при {pct(med.loc['G1_развлечения', 'other_chg'])} в остальных. Предложение: вернуть выгоду в развлечениях (кэшбэк, партнёрские скидки).

**G2. Массовый без приложения: угасание ({grp['G2_массовый_без_приложения']} клиентов).** Падение по всем категориям,
с {med.loc['G2_массовый_без_приложения', 'rate_base']:.1f} до {med.loc['G2_массовый_без_приложения', 'rate_eval']:.1f} операций в месяц (медиана). Самая высокая
вероятность ухода, тот же профиль, что у {excl.loc[0, 'клиентов']} уже ушедших. Канал — СМС или звонок. Предложение: установить
приложение и получить бонус за возврат к покупкам.

**G3. Массовый с приложением: снижение ({grp['G3_массовый_с_приложением']} клиентов).** Падение с {med.loc['G3_массовый_с_приложением', 'rate_base']:.1f}
до {med.loc['G3_массовый_с_приложением', 'rate_eval']:.1f} операций в месяц. Связь через приложение дешёвая: push-кампания
с повышенным кэшбэком в повседневных категориях.
"""))

## 6. Кого не включаем и почему

In [ ]:
display(excl.rename(columns={"кто": "кто", "клиентов": "клиентов"}).style.hide(axis="index"))
display(Markdown("Клиенты, перешедшие с пластика на токен, не выглядят падающими: метрика считает все способы оплаты. "
                 "Под провал «Онлайн-банка» в апреле–мае кампания не нужна, это не поведение клиентов."))

## 7. Что проверили и что не подтвердилось

In [ ]:
display(hyp.style.hide(axis="index"))

## 8. Приложение

### 8.1. Сводка журнала изменений

In [ ]:
s = log[log["action"] != "оставлено"][["field", "issue", "n_rows", "action"]].rename(
    columns={"field": "поле", "issue": "что найдено", "n_rows": "строк", "action": "что сделано"})
display(s.style.hide(axis="index"))
display(Markdown(f"Всего решений в журнале: {len(log)}, из них «оставлено без изменений»: {(log['action'] == 'оставлено').sum()} "
                 "(выбросы сумм, валюты, пропуски дохода и оценки). Полный журнал — в cleaning_log.csv."))

### 8.2. Методика

- Операции: `status = успешно`, дата в 2024-01-01…2025-06-30, клиент есть в справочнике; дубли удалены.
- База клиента: месяцы 2024 года, начиная с месяца **после** `card_open_date` (операций до открытия карты нет;
  первый месяц неполный). `rate_2024` = число операций в базе / число месяцев базы.
- `rate_2025` = число операций за январь–июнь 2025 года / 6. `chg = rate_2025 / rate_2024 − 1`.
- Измеримы клиенты с базой ≥ 1 месяца и `rate_2024 ≥ 1`. Падение: `chg ≤ −0,30`. Сезонный коэффициент
  (январь–июнь к среднему за 2024 год, клиенты с полным годом) = 0,93.
- Развлечения: те же формулы только по `category = Развлечения и досуг` (категория восстановлена по `mcc_code`);
  «остальное» — по всем прочим категориям.
- G1: `ent_2024 ≥ 0,5` и `ent_chg ≤ −0,70` и `other_chg > −0,30`. G2: `segment = Массовый`, `has_mobile_app = Нет`,
  `chg ≤ −0,30`. G3: то же с `has_mobile_app = Да`. Приоритет G1 → G2 → G3. Во всех группах `marketing_opt_in = Да`.
- Центр и разброс скошенных величин — медиана и квартили; оценка сервиса (порядковая шкала) — только медиана.

### 8.3. Использование ИИ

- Использовал ассистента Claude: для разбора задания, кода чтения файлов (кодировки, разделители),
  черновика аудита и чистки, а также структуры отчёта.
- Сам проверил: числа воронки и журнала после запуска ноутбука, что операций до открытия карты действительно нет,
  эффект состава по Казани (разрез сегмент × город), что группы не пересекаются и укладываются в лимиты.
- Пороги (−30%, −70%, 0,5 операции/мес.) выбраны и проверены по распределениям в рабочем ноутбуке.